# MoE XGBoost vs Flat XGBoost — Production Settings

Head-to-head comparison of the User+WC MoE XGBoost model (with decay=0.05)
against the standard flat XGBoost model (no routing, no decay).

**Dataset:** NLR Kestrel, expanded window (Jan 1 – Jun 26 2025, ~2.7M rows)
**Settings:** Production defaults (SVD=256, OHE=2048), 120 windows × 6h, 120-day lookback
**Models:**
- Flat XGBoost (production defaults, no decay)
- MoE XGBoost (user+wallclock routing, decay=0.05)

## 1. Setup

In [ ]:
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from hpc_oda_commons.models.job_runtime_xgboost.model import (
    JobRuntimeXGBoostConfig, JobRuntimeXGBoostModel,
)
from hpc_oda_commons.models.experimental.moe_xgboost_model import (
    MoEXGBoostConfig, MoEXGBoostModel,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

REPO_ROOT = Path.cwd().parent.parent
DATA_PATH = REPO_ROOT / 'workspace' / 'data' / 'datasets' / 'nlr_kestrel' / 'data.parquet'

## 2. Load Data

In [ ]:
table = pq.read_table(DATA_PATH)
lo = datetime(2025, 1, 1, tzinfo=timezone.utc)
hi = datetime(2025, 6, 26, tzinfo=timezone.utc) + timedelta(days=1)
sc = table.column('submit_time')
ec = table.column('end_time')
mask = pc.and_(
    pc.less(sc, pa.scalar(hi, type=sc.type)),
    pc.greater_equal(ec, pa.scalar(lo, type=ec.type)),
)
df = table.filter(mask).to_pandas()
rows = df.to_dict('records')
print(f'Loaded: {len(rows):,} rows')
print(f'Span: {(df["submit_time"].max() - df["submit_time"].min()).days} days')

## 3. Configuration

Production settings: SVD=256, OHE=2048, 120 windows, 120-day lookback.

In [ ]:
N_WINDOWS = 120
TEST_WINDOW_HOURS = 6
TRAINING_LOOKBACK_DAYS = 120

# Production SVD/OHE
MAX_SVD = 256
MAX_OHE = 2048

print(f'Windows: {N_WINDOWS} x {TEST_WINDOW_HOURS}h = {N_WINDOWS * TEST_WINDOW_HOURS / 24:.0f} days test')
print(f'Lookback: {TRAINING_LOOKBACK_DAYS} days')
print(f'SVD: {MAX_SVD}, OHE: {MAX_OHE}')

## 4. Run Flat XGBoost (baseline)

In [ ]:
print('Running Flat XGBoost (no MoE, no decay)...')
flat_config = JobRuntimeXGBoostConfig(
    n_windows=N_WINDOWS,
    test_window_hours=TEST_WINDOW_HOURS,
    training_lookback_days=TRAINING_LOOKBACK_DAYS,
    max_svd_components=MAX_SVD,
    target_max_one_hot_width=MAX_OHE,
)
flat_model = JobRuntimeXGBoostModel(flat_config)

flat_start = time.time()
flat_result = flat_model.evaluate(rows, verbose=True)
flat_time = (time.time() - flat_start) / 60

print(f'\nFlat XGBoost: MAE={flat_result["mae"]:,.0f}s, RMSE={flat_result["rmse"]:,.0f}s')
print(f'Scored: {flat_result["summary"]["rows_scored"]:,}')
print(f'Time: {flat_time:.1f}min', flush=True)

## 5. Run MoE XGBoost (user+wallclock routing, decay=0.05)

In [ ]:
print('Running MoE XGBoost (user+WC routing, decay=0.05)...')
moe_config = MoEXGBoostConfig(
    n_windows=N_WINDOWS,
    test_window_hours=TEST_WINDOW_HOURS,
    training_lookback_days=TRAINING_LOOKBACK_DAYS,
    max_svd_components=MAX_SVD,
    target_max_one_hot_width=MAX_OHE,
    time_decay_rate=0.05,
    estimator_n_jobs=12,
)
moe_model = MoEXGBoostModel(moe_config)

moe_start = time.time()
moe_result = moe_model.evaluate(rows, verbose=True)
moe_time = (time.time() - moe_start) / 60

print(f'\nMoE XGBoost: MAE={moe_result["mae"]:,.0f}s, RMSE={moe_result["rmse"]:,.0f}s')
print(f'Scored: {moe_result["summary"]["rows_scored"]:,}')
print(f'Time: {moe_time:.1f}min', flush=True)

## 6. Results

In [ ]:
print('=' * 60)
print('RESULTS: Flat XGBoost vs MoE XGBoost')
print('=' * 60)
print(f'Settings: SVD={MAX_SVD}, OHE={MAX_OHE}, windows={N_WINDOWS}, lookback={TRAINING_LOOKBACK_DAYS}d')
print()
print(f'{"Model":<30} {"MAE":>10} {"RMSE":>10} {"Scored":>10}')
print('-' * 65)
print(f'{"Flat XGBoost":<30} {flat_result["mae"]:>10,.0f}s {flat_result["rmse"]:>10,.0f}s {flat_result["summary"]["rows_scored"]:>10,}')
print(f'{"MoE XGBoost (decay=0.05)":<30} {moe_result["mae"]:>10,.0f}s {moe_result["rmse"]:>10,.0f}s {moe_result["summary"]["rows_scored"]:>10,}')
print()

mae_imp = (moe_result['mae'] - flat_result['mae']) / flat_result['mae'] * 100
rmse_imp = (moe_result['rmse'] - flat_result['rmse']) / flat_result['rmse'] * 100
print(f'MAE improvement:  {mae_imp:+.1f}%')
print(f'RMSE improvement: {rmse_imp:+.1f}%')
print(f'\nFlat time: {flat_time:.0f}min, MoE time: {moe_time:.0f}min', flush=True)

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

models = ['Flat XGBoost', 'MoE XGBoost\n(decay=0.05)']
maes = [flat_result['mae'], moe_result['mae']]
rmses = [flat_result['rmse'], moe_result['rmse']]
colors = ['#4a90d9', '#e74c3c']

axes[0].bar(models, maes, color=colors)
axes[0].set_ylabel('MAE (seconds)')
axes[0].set_title('Mean Absolute Error')
for i, v in enumerate(maes):
    axes[0].text(i, v + max(maes) * 0.02, f'{v:,.0f}s', ha='center', fontsize=10)

axes[1].bar(models, rmses, color=colors)
axes[1].set_ylabel('RMSE (seconds)')
axes[1].set_title('Root Mean Squared Error')
for i, v in enumerate(rmses):
    axes[1].text(i, v + max(rmses) * 0.02, f'{v:,.0f}s', ha='center', fontsize=10)

plt.suptitle(f'Flat XGBoost vs MoE XGBoost (Production Settings, {flat_result["summary"]["rows_scored"]:,} scored)', y=1.02)
plt.tight_layout()
plt.show()

## 7. Per-Bin Breakdown (MoE)

In [ ]:
print(f'{"Bin":<35} {"Rows":>8} {"Scored":>8} {"MAE":>10} {"RMSE":>10}')
print('-' * 75)
for b in moe_result['summary']['bin_details']:
    if b['scored'] > 0:
        print(f'{b["bin"]:<35} {b["rows"]:>8,} {b["scored"]:>8,} {b["mae"]:>10,.0f}s {b["rmse"]:>10,.0f}s')
    else:
        print(f'{b["bin"]:<35} {b["rows"]:>8,} {b["scored"]:>8,} {"—":>10} {"—":>10}')